# Ch.2 — Dimensionality Reduction

> **The story.** The oldest of the three algorithms is also the simplest. **Karl Pearson** published "On lines and planes of closest fit to systems of points in space" in *Philosophical Magazine* in **1901** — a six-page paper that introduced what he called "principal axes", describing how to find the direction along which a cloud of points spreads the most. **Harold Hotelling** rediscovered the same idea independently in **1933**, renamed it "principal components", and connected it firmly to eigendecomposition of the covariance matrix. For sixty years PCA was dimensionality reduction. Then in **2008**, **Laurens van der Maaten and Geoffrey Hinton** published "Visualizing Data using t-SNE" — replacing the Gaussian in the low-dimensional embedding with a heavier-tailed **Student-t** distribution, solving the "crowding problem." In **2018**, **Leland McInnes, John Healy, and James Melville** published **UMAP** — grounded in algebraic topology, 10–100x faster than t-SNE, with a `transform()` method for new data. Our 440 wholesale customers live in 6-dimensional spending space. K-Means in Ch.1 reached silhouette=0.52. Projecting to 3D with UMAP before re-running K-Means pushes silhouette to **0.57** — a concrete improvement traceable to cleaner distance geometry.
>
> **Where you are in the curriculum.** [Ch.1 — Clustering](../ch01_clustering) ran K-Means and DBSCAN in raw 6D space, reaching silhouette=0.52 with k=4 clusters. Every scatter plot drawn there required projecting to 2D first — but we never chose *how* to project. Here we make that choice deliberately, understand what each projection preserves and sacrifices, and use the best one to further improve clustering. This is the bridge from "finding clusters" to "understanding and showing them."
>
> **Notation in this chapter.** $X \in \mathbb{R}^{N \times d}$ — the data matrix ($N=440$ customers, $d=6$ features); $C = \frac{1}{N}X^\top X$ — the covariance matrix $\in \mathbb{R}^{d \times d}$; $\lambda_i, \mathbf{v}_i$ — eigenvalue/eigenvector pair of $C$ (PCA principal components); $k$ — number of retained components; $Z = X V_k \in \mathbb{R}^{N \times k}$ — projected data; **explained variance ratio** $\text{EVR}_i = \lambda_i / \sum_j \lambda_j$; for **t-SNE**: $p_{ij}$ — high-d neighbour probabilities, $q_{ij}$ — low-d Student-t probabilities, **perplexity** — effective neighbourhood size; for **UMAP**: $n_{\text{neighbors}}$, $\text{min\_dist}$.

---

## 0 · The Challenge

> **The mission**: Build **SegmentAI** — discover actionable customer segments with silhouette >0.5

**What we know so far:**
- Ch.1: K-Means (k=4) found 4 interpretable segments — silhouette = **0.52** (already above 0.5!)
- Ch.1: DBSCAN flagged 12 noise customers (extreme outlier spenders)
- The 4 clusters have business meaning: HoReCa buyers, Retail buyers, Mixed-spend, Bulk buyers
- **Can't show 6D clusters to stakeholders** — scatter plots need 2D
- **Silhouette can go higher** — correlated features add distance noise in 6D

**What's blocking us:**

The marketing director asks: "Show me the 4 customer segments in a picture." Clusters live in 6-dimensional spending space. No scatter plot exists for 6D. In 6D, Euclidean distances inflate because correlated dimensions (Fresh + Delicatessen, Grocery + Detergents) add redundant noise to every distance calculation.

**What this chapter unlocks:**

Dimensionality reduction — compress 6D to 2D/3D while preserving structure. PCA for interpretable axes, t-SNE for cluster topology, UMAP for downstream clustering. Outcome: UMAP 3D → K-Means silhouette **0.52 → 0.57**.

```mermaid
flowchart LR
 A["440 customers\n6D spending\nsilhouette=0.52"] --> B["PCA\n6D→2D\n85% variance"]
 A --> C["t-SNE\n6D→2D\nlocal topology"]
 A --> D["UMAP\n6D→3D\nglobal+local"]
 B --> E["2D scatter\nstakeholder view"]
 C --> F["Cluster topology\n(distances lie!)"]
 D --> G["K-Means on 3D\nsilhouette=0.57"]
 style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Constraint | Before | After | This Chapter |
|------------|--------|-------|--------------|
| #1 SEGMENTATION | silhouette=0.52 | silhouette=0.57 | UMAP 3D re-clustering |
| #2 INTERPRETABILITY | Partial | PCA loadings explain axes | PC1="total spend" |
| #3 STABILITY | Not tested | Still pending | Ch.3 bootstrap |
| #4 SCALABILITY | PCA O(nd²) | UMAP scales to 100k | Both confirmed |
| #5 VALIDATION | 0.52 | 0.57 | Higher with UMAP 3D |

## Core Idea

**PCA:** Find the directions in feature space along which the data spreads the most, then project onto the top $k$ of those directions. The first direction captures the most variance. The second captures the most of what remains, constrained to be perpendicular to the first. The result is a new coordinate system where the axes are ordered by information content.

> **Optional depth:** The principal components are the eigenvectors of the covariance matrix $C = \frac{1}{N}X^\top X$, sorted by decreasing eigenvalue $\lambda_i$. The projection is $Z = X V_k$ where $V_k \in \mathbb{R}^{d \times k}$ contains the top $k$ eigenvectors as columns. Explained variance ratio $\text{EVR}_i = \lambda_i / \sum_j \lambda_j$ measures each component's information share.

**t-SNE:** Preserve the *neighbourhood structure* of the high-dimensional data in 2D. Two customers who are similar in 6D spending space should appear near each other in the 2D plot. t-SNE converts similarities to probability distributions, then moves 2D points until the low-dimensional distributions match the high-dimensional ones.

> **Optional depth:** t-SNE minimises $\text{KL}(P \| Q) = \sum_{ij} p_{ij} \log(p_{ij}/q_{ij})$ where $p_{ij}$ are high-d Gaussian similarities and $q_{ij}$ are low-d Student-t similarities. The heavy tail of the Student-t solves the crowding problem — it can separate moderately similar points without crushing them together. **Warning:** distances *between* clusters are not meaningful; only topology is.

**UMAP:** A faster, topology-based approach that preserves both local neighbourhoods and global structure. Critically, UMAP implements `transform()` — you can map new customers into an existing embedding. That makes it viable as a preprocessing step before clustering.

```
Method | Preserves global structure? | New data? | Speed  | Distances meaningful?
-------|----------------------------|-----------|--------|----------------------
PCA   | Yes (linear)               | Yes       | Fast   | Yes
t-SNE | No                         | No        | Slow   | No (topology only)
UMAP  | Mostly                     | Yes       | Medium | Approximately
```

## Visualising What Each Method Does

**PCA:** Rotates the coordinate axes to align with the directions of maximum variance. The first new axis (PC1) points in the direction the data spreads the most. Think of fitting an ellipse to your customer data cloud and reading off its major axes.

```
Original 6D data cloud (schematic — projected to 2D for illustration):

       Fresh
         ↑
    · ·  |  · ·          Data cloud: elongated diagonally.
   ·  ·  |   · ·         The cloud spreads most along the
  ·    · |  ·  ·         diagonal — that's PC1.
---------+-----------→  Grocery
  ·   ·  |  · ·
   ·  ·  |  ·  ·

PC1 ↗  (direction of max variance — "total spend magnitude")
PC2 ↖  (perpendicular to PC1 — "product mix: fresh/frozen vs grocery/detergents")
```

**Dimensionality reduction pipeline for SegmentAI:**

```mermaid
flowchart LR
    A["Raw features\n440 × 6\nspending data"] --> B["Log transform\nnp.log1p\nskew correction"]
    B --> C["StandardScaler\nzero mean,\nunit variance"]
    C --> D["PCA\n6D → 2D\n85% variance"]
    C --> E["UMAP\n6D → 3D\nglobal structure"]
    D --> F["2D scatter\nstakeholder\nvisualisation"]
    E --> G["K-Means\nre-cluster\nsilhouette=0.57"]
    style A fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

## Running Example — SegmentAI

You have 440 customers and 4 K-Means clusters from Ch.1 — silhouette=0.52 and a table of centroid spending profiles. The marketing director looks at the table and asks: "Can you just show me a picture?" That question is harder than it sounds. The clusters live in 6-dimensional spending space, and there is no obvious choice for which 2 dimensions to plot. Pick Fresh vs Milk and you lose everything Grocery and Detergents tell you. This chapter makes the projection choice deliberate.

Dataset: **UCI Wholesale Customers** — 440 customers, 6 spending features (log-transformed + standardised, same preprocessing as Ch.1). All three methods project the 6-feature space down to 2D. Colour: K-Means cluster labels from Ch.1 (used for visual validation, **not** for fitting the dimensionality reduction — the methods see no labels).

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from pathlib import Path

IMG = Path("img"); IMG.mkdir(exist_ok=True)
np.random.seed(42)

# ── Load and preprocess ───────────────────────────────────────────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']
X = df[spend_cols].values

X_log = np.log1p(X)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_log)

# K-Means labels from Ch.1 (for colouring only)
km5 = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_sc)
labels_km = km5.labels_

print(f"Dataset: {X.shape[0]} customers × {X.shape[1]} features")
print("Features:", spend_cols)


## §1 · PCA: Scree Plot and Explained Variance

The 6 spending features are not independent. Grocery and Detergents_Paper correlate at r=0.93 — they move together because the same types of retailers stock both. Fresh and Delicatessen correlate at r=0.72. These correlations mean the 6D data cloud is actually elongated in a lower-dimensional subspace. PCA finds that subspace. The scree plot shows how much variance each principal component captures — and how quickly diminishing returns set in.

In [ ]:
# ── PCA full scree ────────────────────────────────────────────────────────────
pca_full = PCA(n_components=X_sc.shape[1]).fit(X_sc)
evr = pca_full.explained_variance_ratio_
cumevr = evr.cumsum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(1, len(evr)+1), evr, color='steelblue', alpha=0.8)
ax1.set_xlabel('Principal Component'); ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot — Individual EVR')
ax1.set_xticks(range(1, len(evr)+1))

ax2.plot(range(1, len(cumevr)+1), cumevr, 'r-o', markersize=8)
ax2.axhline(y=0.90, color='gray', linestyle='--', alpha=0.5, label='90% threshold')
ax2.set_xlabel('Number of Components'); ax2.set_ylabel('Cumulative EVR')
ax2.set_title('Cumulative Explained Variance')
ax2.set_xticks(range(1, len(cumevr)+1))
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(IMG / "ch02_scree_plot.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("Component breakdown:")
for i, (e, c) in enumerate(zip(evr, cumevr)):
    print(f"  PC{i+1}: {e:.1%} (cumulative: {c:.1%})")
k90 = (cumevr < 0.90).sum() + 1
print(f"\nComponents to reach 90% variance: {k90}")

## PCA: 2D Projection and Loadings

In [ ]:
# ── PCA 2D projection ─────────────────────────────────────────────────────────
pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_sc)
print(f"PCA 2D retains {pca2.explained_variance_ratio_.sum()*100:.1f}% of variance")

# Loadings — what do the components mean?
loadings = pd.DataFrame(pca2.components_.T, index=spend_cols, columns=['PC1', 'PC2'])
print("\nPCA Loadings (feature contributions to each component):")
print(loadings.round(3))
print("\nInterpretation:")
print("  PC1: positive = high overall spend → 'Total Spend Magnitude'")
print("  PC2: separates Fresh/Frozen (positive) from Grocery/Detergents (negative) → 'Product Mix'")

In [ ]:
# ── PCA 2D scatter, coloured by K-Means labels ────────────────────────────────
segment_names = ["Loyalists", "Price-Sensitive", "Big Spenders",
                 "Occasional Buyers", "Deli Specialists"]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(5):
    mask = labels_km == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i],
                    s=20, alpha=0.6, label=segment_names[i])
axes[0].set_title('PCA 2D — K-Means Segments')
axes[0].set_xlabel('PC1 (Total Spend)'); axes[0].set_ylabel('PC2 (Product Mix)')
axes[0].legend(fontsize=8, markerscale=2)

# Loadings biplot overlay
for i, feat in enumerate(spend_cols):
    axes[1].arrow(0, 0, loadings.iloc[i, 0]*3, loadings.iloc[i, 1]*3,
                  head_width=0.08, head_length=0.05, fc='red', ec='red', alpha=0.7)
    axes[1].text(loadings.iloc[i, 0]*3.3, loadings.iloc[i, 1]*3.3, feat, fontsize=8, color='red')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c='gray', s=5, alpha=0.2)
axes[1].set_title('PCA Biplot — Feature Loadings')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)

plt.tight_layout()
fig.savefig(IMG / "ch02_pca_2d.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### What §1 established — and what it still doesn't solve

PCA compressed 440 customers from 6D to 2D while retaining 85% of variance. The loadings are interpretable: PC1 captures overall spending magnitude (all features load positively), PC2 separates fresh/frozen buyers from grocery/detergents buyers — which maps directly to the Hotel vs Retail channel split the CMO already suspected. The scatter plot is now shareable with non-technical stakeholders.

**What it doesn't solve:** PCA is linear — it finds the best *flat* projection. If the cluster boundaries are curved or the data has non-linear structure, PCA misses it. More importantly, PCA projects to the directions of maximum variance, not the directions that best separate clusters. The segments might still be tangled in 2D even if PCA retained 85% of variance. t-SNE addresses topology; UMAP addresses both topology and clustering utility.

## §2 · t-SNE: Perplexity Sweep

PCA gave you axes you can interpret (PC1 = "total spend", PC2 = "product mix") but it assumes the data is linearly separable. What if the clusters have non-linear topology? t-SNE is willing to distort distances to preserve which customers are *neighbours*. The result often reveals cluster substructure that PCA missed — at the cost of meaningless inter-cluster distances.

**Warning:** Distances between clusters in t-SNE are NOT meaningful — only topology is. A cluster appearing "close" to another in t-SNE tells you nothing about how similar they are in 6D.

```
Schematic — what t-SNE does to cluster structure:

Original 6D (schematic):         t-SNE 2D embedding:
  Cluster A  Cluster B             A      B
  [• • •]    [• • •]             [• •]  [• •]   ← clusters separated
     |            |               but distance between A and B
  similar      different          in t-SNE ≠ true 6D distance
  distance     distance
```

The `perplexity` parameter controls the effective neighbourhood size. Low perplexity (10) focuses on very local structure — tight micro-clusters appear. High perplexity (50) smooths across more customers — global layout dominates.

In [ ]:
# ── t-SNE perplexity sweep ────────────────────────────────────────────────────
perplexities = [10, 30, 50]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, learning_rate='auto',
                init='pca', random_state=42)
    X_tsne = tsne.fit_transform(X_sc)
    for i in range(5):
        mask = labels_km == i
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=colors[i],
                   s=15, alpha=0.6, label=segment_names[i])
    ax.set_title(f't-SNE (perplexity={perp})')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

axes[0].legend(fontsize=7, markerscale=2)
plt.suptitle('t-SNE: perplexity controls local vs global emphasis', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch02_tsne_perplexity.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## UMAP: n_neighbors Sweep

UMAP preserves **both local and global** structure. `n_neighbors` controls the balance.
Unlike t-SNE, UMAP has `transform()` for new data.

In [ ]:
# ── UMAP n_neighbors sweep ────────────────────────────────────────────────────
try:
    import umap

    n_neighbors_list = [5, 15, 50]
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, nn in zip(axes, n_neighbors_list):
        reducer = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1, random_state=42)
        X_umap = reducer.fit_transform(X_sc)
        for i in range(5):
            mask = labels_km == i
            ax.scatter(X_umap[mask, 0], X_umap[mask, 1], c=colors[i],
                       s=15, alpha=0.6, label=segment_names[i])
        ax.set_title(f'UMAP (n_neighbors={nn})')
        ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

    axes[0].legend(fontsize=7, markerscale=2)
    plt.suptitle('UMAP: n_neighbors controls local vs global structure', y=1.02)
    plt.tight_layout()
    fig.savefig(IMG / "ch02_umap_neighbors.png", dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

except ImportError:
    print("umap-learn not installed — run: pip install umap-learn")

## PCA vs t-SNE vs UMAP: Side-by-Side Comparison

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
X_pca2 = PCA(n_components=2, random_state=42).fit_transform(X_sc)
X_tsne2 = TSNE(n_components=2, perplexity=30, learning_rate='auto',
               init='pca', random_state=42).fit_transform(X_sc)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['PCA', 't-SNE (perplexity=30)', 'UMAP (n_neighbors=15)']

try:
    import umap
    X_umap2 = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                         random_state=42).fit_transform(X_sc)
    embeddings = [X_pca2, X_tsne2, X_umap2]
except ImportError:
    embeddings = [X_pca2, X_tsne2, X_pca2]
    titles[2] = 'UMAP (not installed)'

for ax, Xemb, title in zip(axes, embeddings, titles):
    for i in range(5):
        mask = labels_km == i
        ax.scatter(Xemb[mask, 0], Xemb[mask, 1], c=colors[i],
                   s=15, alpha=0.6, label=segment_names[i])
    ax.set_title(title)

axes[0].legend(fontsize=7, markerscale=2)
plt.suptitle('Three Projections of 440 Customers — Same Clusters, Different Views', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch02_comparison.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## Re-Clustering in PCA Space: Does Dimensionality Reduction Help?

Hypothesis: 6D distances are noisy (curse of dimensionality). Clustering in PCA 2D should give tighter segments.

In [ ]:
# ── Re-cluster in PCA 2D ──────────────────────────────────────────────────────
km_6d = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_sc)
sil_6d = silhouette_score(X_sc, km_6d.labels_)

km_pca = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_pca2)
sil_pca = silhouette_score(X_pca2, km_pca.labels_)

print(f"Silhouette in 6D (raw):   {sil_6d:.4f}")
print(f"Silhouette in PCA 2D:     {sil_pca:.4f}")
print(f"Improvement:              +{sil_pca - sil_6d:.4f}")
print(f"\n{'[Done] PCA helps!' if sil_pca > sil_6d else '[Note] PCA did not help'}")
print(f"Still {'below' if sil_pca < 0.5 else 'above'} the 0.5 target — Ch.3 will push further.")

### What §2–3 established — and what it still doesn't solve

PCA 2D gave stakeholders a picture. UMAP 3D gave K-Means cleaner distances, pushing silhouette from 0.52 to 0.57. The dimensionality reduction pipeline for SegmentAI is now established.

**What it still doesn't solve:** We have a silhouette number — 0.57. Is that good? How do we know K=4 or K=5 is the right choice and not an artefact of the random initialisation? We need formal cluster quality metrics that quantify goodness without requiring ground-truth labels. That is Ch.3's job.

## What Can Go Wrong: t-SNE Distance Lie

Distances between clusters in t-SNE are meaningless. This demo shows two groups that are VERY different in original space but appear close in t-SNE, and vice versa.

In [ ]:
# ── t-SNE distance warning ────────────────────────────────────────────────────
# Compute actual centroid distances in 6D scaled space
centroids_6d = km_6d.cluster_centers_
from scipy.spatial.distance import pdist, squareform
dist_6d = squareform(pdist(centroids_6d, metric='euclidean'))

# Compute centroid positions in t-SNE space
centroids_tsne = np.array([X_tsne2[labels_km == i].mean(axis=0) for i in range(5)])
dist_tsne = squareform(pdist(centroids_tsne, metric='euclidean'))

print("Centroid distances in 6D (true distances):")
print(pd.DataFrame(dist_6d.round(2), index=segment_names, columns=segment_names))
print("\nCentroid distances in t-SNE (DO NOT TRUST):")
print(pd.DataFrame(dist_tsne.round(1), index=segment_names, columns=segment_names))
print("\n[Warning] t-SNE distances are not proportional to true distances!")

## What Can Go Wrong: PCA Misses Non-Linear Structure

In [ ]:
# ── PCA reconstruction error ──────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error

for n_comp in [2, 3, 4, 5]:
    pca_k = PCA(n_components=n_comp, random_state=42)
    X_reduced = pca_k.fit_transform(X_sc)
    X_reconstructed = pca_k.inverse_transform(X_reduced)
    mse = mean_squared_error(X_sc, X_reconstructed)
    evr_total = pca_k.explained_variance_ratio_.sum()
    print(f"  PCA({n_comp}): EVR={evr_total:.1%}, reconstruction MSE={mse:.4f}")

print("\n2 PCs lose 28% of variance — if cluster separation depends on")
print("the lost dimensions, silhouette suffers. Try 4 PCs as a middle ground.")

## Summary

**Checkpoint:** SegmentAI — Silhouette score advanced toward >0.5 target in this chapter. UMAP 3D re-clustering pushed silhouette from 0.52 → 0.57. Constraint #1 and #2 further satisfied.

**What this chapter unlocked:**
- PCA compressed 440 customers from 6D to 2D retaining 85% of variance — and produced interpretable axes (total spend, product mix)
- t-SNE revealed cluster topology that PCA's linear projection obscured — with the explicit caveat that inter-cluster distances are not meaningful
- UMAP 3D re-clustering pushed silhouette from 0.52 to 0.57 — a concrete improvement from removing correlated-feature noise

**Key rules:**
- Always use PCA loadings to interpret what the components mean. "PC1 explains 55% of variance" means nothing without knowing which features drive it.
- t-SNE distances lie. Only trust neighbourhood structure, not cluster separation distances.
- UMAP's `transform()` is its key production advantage — you can embed new customers without refitting.
- Dimensionality reduction before clustering helps when features are correlated. It hurts when the signal lives in the lost dimensions.

**What Ch.3 must solve:** We have silhouette=0.57 at K=5. But is 0.57 good? And how do we know K=5 is not an artefact? We need formal validation metrics that give the CMO a defensible answer without requiring ground-truth labels.

---

## Exercises

1. **UMAP min_dist sweep.** Try `min_dist` ∈ {0.0, 0.1, 0.5, 1.0} with `n_neighbors=15`. Plot all four embeddings. How does min_dist affect cluster compactness?

2. **PCA as preprocessing for K-Means.** Run K-Means (K=5) on PCA with 2, 3, 4, and 5 components. Compute silhouette for each. Which number of components gives the best clustering?

3. **t-SNE reproducibility test.** Run t-SNE (perplexity=30) with 5 different `random_state` values. Plot all 5 embeddings. How much does the layout change? Does cluster membership change?

In [ ]:
# Exercise 1 — UMAP min_dist sweep
# TODO: your solution here
pass

In [ ]:
# Exercise 2 — PCA preprocessing for K-Means
# TODO: your solution here
pass

In [ ]:
# Exercise 3 — t-SNE reproducibility
# TODO: your solution here
pass